In [1]:
"""
PR2 unit tests — MoiraicModule decode-with-cache equivalence.

Invariants verified:
  T1: full(seq) == prefill(seq) at all positions.
  T2: full(seq) == decode(seq[suffix], past_cache=cache_from_prefill) at suffix.
  T3: After a simulated AR commit (write a value into the prediction window,
      flip prediction_mask), decode(updated_suffix, past_cache=...) matches
      a fresh full forward over the updated sequence at the suffix.
  T4: Cache structure has the expected keys/shapes, and ctx_len equals the
      number of context tokens (uniform across batch).
  T5: Multivariate (D=2) variant of T2 + T3.
"""
import torch

from uni2ts.model.moiraic.module import MoiraicModule

torch.manual_seed(0)
device = "cuda:7" if torch.cuda.is_available() else "cpu"
ATOL = 1e-4


def make_module(num_predict_token=1, patch_size=8, max_seq_len=64):
    return (
        MoiraicModule(
            d_model=64, d_ff=128, num_layers=2,
            patch_size=patch_size, max_seq_len=max_seq_len,
            attn_dropout_p=0.0, dropout_p=0.0,
            scaling=True, num_predict_token=num_predict_token,
        )
        .to(device)
        .eval()
    )


def make_packed_inputs(B, D, ctx_tokens, pred_tokens, P, device):
    """
    Build a moiraic-shaped packed sequence:

        layout = [ var0_ctx | var1_ctx | ... | var0_pred | var1_pred | ... ]

    Uses a contiguous context prefix and contiguous prediction suffix per
    variate, matching MoiraicForecast._convert.
    """
    S = D * (ctx_tokens + pred_tokens)
    target = torch.randn(B, S, P, device=device)
    observed_mask = torch.ones(B, S, P, dtype=torch.bool, device=device)
    sample_id = torch.ones(B, S, dtype=torch.long, device=device)
    prediction_mask = torch.zeros(B, S, dtype=torch.bool, device=device)

    var_id_rows, time_id_rows = [], []
    # Context block: per variate, ctx_tokens positions with time_id 0..ctx-1.
    for v in range(D):
        var_id_rows.append(torch.full((ctx_tokens,), v, dtype=torch.long, device=device))
        time_id_rows.append(torch.arange(ctx_tokens, device=device))
    # Prediction block: per variate, pred_tokens positions with time_id ctx..ctx+pred-1.
    for v in range(D):
        var_id_rows.append(torch.full((pred_tokens,), v, dtype=torch.long, device=device))
        time_id_rows.append(
            torch.arange(ctx_tokens, ctx_tokens + pred_tokens, device=device)
        )
    variate_id = torch.cat(var_id_rows).unsqueeze(0).expand(B, -1).contiguous()
    time_id = torch.cat(time_id_rows).unsqueeze(0).expand(B, -1).contiguous()

    # Mark prediction suffix (per variate) and zero out its target/observed.
    ctx_total = D * ctx_tokens
    prediction_mask[:, ctx_total:] = True
    target[:, ctx_total:, :] = 0.0
    observed_mask[:, ctx_total:, :] = False

    return target, observed_mask, sample_id, time_id, variate_id, prediction_mask, ctx_total


def slice_suffix(*tensors, start):
    return tuple(t[..., start:, :] if t.ndim == 3 else t[..., start:] for t in tensors)


In [2]:
# ---------------------------------------------------------------------------
# T1 + T4 — prefill equals full forward; cache shape sanity
# ---------------------------------------------------------------------------
def test_prefill_returns_unchanged_outputs_and_well_formed_cache():
    module = make_module()
    B, D, ctx, pred, P = 2, 1, 5, 3, module.patch_size
    target, obs, sid, tid, vid, pmask, ctx_total = make_packed_inputs(
        B, D, ctx, pred, P, device
    )

    with torch.no_grad():
        full = module(target, obs, sid, tid, vid, pmask, training_mode=False)
        pref, cache = module(
            target, obs, sid, tid, vid, pmask,
            training_mode=False, return_cache=True,
        )

    assert torch.allclose(full, pref, atol=ATOL), \
        f"prefill diverged from full: {(full - pref).abs().max().item():.2e}"

    # Cache schema
    for k in ("layer_kvs", "kv_var_id", "kv_time_id", "kv_sample_id",
              "variate_loc", "variate_scale", "ctx_len"):
        assert k in cache, f"missing cache key: {k}"
    assert cache["ctx_len"] == ctx_total
    assert len(cache["layer_kvs"]) == module.num_layers
    for k_, v_ in cache["layer_kvs"]:
        assert k_.shape[-2] == ctx_total and v_.shape[-2] == ctx_total, \
            f"cache K/V seq len {k_.shape[-2]} != ctx_total {ctx_total}"
    assert cache["kv_var_id"].shape == (B, ctx_total)
    assert cache["variate_loc"].shape == (B, D, 1)
    assert cache["variate_scale"].shape == (B, D, 1)
    print("✓ test_prefill_returns_unchanged_outputs_and_well_formed_cache")


test_prefill_returns_unchanged_outputs_and_well_formed_cache()

✓ test_prefill_returns_unchanged_outputs_and_well_formed_cache


In [3]:
# ---------------------------------------------------------------------------
# T2 — decode on prediction suffix matches one-shot forward at the suffix
# ---------------------------------------------------------------------------
def test_decode_matches_full_on_suffix_univariate():
    module = make_module()
    B, D, ctx, pred, P = 2, 1, 5, 3, module.patch_size
    target, obs, sid, tid, vid, pmask, ctx_total = make_packed_inputs(
        B, D, ctx, pred, P, device
    )

    with torch.no_grad():
        full = module(target, obs, sid, tid, vid, pmask, training_mode=False)
        _, cache = module(
            target, obs, sid, tid, vid, pmask,
            training_mode=False, return_cache=True,
        )
        decoded = module(
            target[:, ctx_total:],
            obs[:, ctx_total:],
            sid[:, ctx_total:],
            tid[:, ctx_total:],
            vid[:, ctx_total:],
            pmask[:, ctx_total:],
            training_mode=False,
            past_cache=cache,
        )

    diff = (full[:, ctx_total:] - decoded).abs().max().item()
    assert diff < ATOL, f"decode mismatch: {diff:.2e}"
    print("✓ test_decode_matches_full_on_suffix_univariate")


test_decode_matches_full_on_suffix_univariate()

✓ test_decode_matches_full_on_suffix_univariate


In [13]:
# ---------------------------------------------------------------------------
# T3 — decode after a simulated AR commit matches full forward
#       on the updated state
# ---------------------------------------------------------------------------
def test_decode_after_ar_commit_matches_frozen_scaler_full():
    """
    After an AR commit, decode-with-cache must match a full forward that
    uses the *original* prediction_mask for scaling (i.e. holds loc/scale
    frozen at context-only) but the *committed* target / observed_mask
    everywhere else. This is the invariant the context-only cache enforces.
    """
    module = make_module()
    B, D, ctx, pred, P = 2, 1, 5, 3, module.patch_size
    target, obs, sid, tid, vid, pmask, ctx_total = make_packed_inputs(
        B, D, ctx, pred, P, device
    )

    with torch.no_grad():
        _, cache = module(
            target, obs, sid, tid, vid, pmask,
            training_mode=False, return_cache=True,
        )

        # Simulate AR commit: write predicted value, mark observed, flip pmask.
        target_c = target.clone()
        obs_c = obs.clone()
        pmask_c = pmask.clone()
        for v in range(D):
            commit_pos = ctx_total + v * pred
            target_c[:, commit_pos, :] = torch.randn(B, P, device=device)
            obs_c[:, commit_pos, :] = True
            pmask_c[:, commit_pos] = False

        # Reference: full forward with COMMITTED target/obs but ORIGINAL pmask,
        # so the scaler still sees only the context — matching the cache's
        # frozen-scaler semantics.
        full_frozen = module(
            target_c, obs_c, sid, tid, vid, pmask,   # <-- original pmask
            training_mode=False,
        )

        # Decode: feed only the suffix slice with the committed pmask_c.
        decoded = module(
            target_c[:, ctx_total:],
            obs_c[:, ctx_total:],
            sid[:, ctx_total:],
            tid[:, ctx_total:],
            vid[:, ctx_total:],
            pmask_c[:, ctx_total:],
            training_mode=False,
            past_cache=cache,
        )

    diff = (full_frozen[:, ctx_total:] - decoded).abs().max().item()
    assert diff < ATOL, f"frozen-scaler AR-commit decode mismatch: {diff:.2e}"
    print("✓ test_decode_after_ar_commit_matches_frozen_scaler_full")

test_decode_after_ar_commit_matches_frozen_scaler_full()

✓ test_decode_after_ar_commit_matches_frozen_scaler_full


In [9]:
# ---------------------------------------------------------------------------
# T5 — multivariate (D=2) decode equivalence (with and without commit)
# ---------------------------------------------------------------------------
def test_decode_multivariate():
    module = make_module()
    B, D, ctx, pred, P = 2, 2, 4, 3, module.patch_size
    target, obs, sid, tid, vid, pmask, ctx_total = make_packed_inputs(
        B, D, ctx, pred, P, device
    )

    with torch.no_grad():
        full = module(target, obs, sid, tid, vid, pmask, training_mode=False)
        _, cache = module(
            target, obs, sid, tid, vid, pmask,
            training_mode=False, return_cache=True,
        )
        decoded = module(
            target[:, ctx_total:],
            obs[:, ctx_total:],
            sid[:, ctx_total:],
            tid[:, ctx_total:],
            vid[:, ctx_total:],
            pmask[:, ctx_total:],
            training_mode=False,
            past_cache=cache,
        )

    diff = (full[:, ctx_total:] - decoded).abs().max().item()
    assert diff < ATOL, f"multivariate decode mismatch: {diff:.2e}"

    # Sanity check: per-variate loc/scale lookup tables differ across variates
    # (so the scatter actually populated independent entries).
    assert not torch.allclose(cache["variate_loc"][:, 0], cache["variate_loc"][:, 1]), \
        "variate_loc collapsed across variates — scatter is wrong"
    print("✓ test_decode_multivariate")


test_decode_multivariate()

✓ test_decode_multivariate


In [10]:
# ---------------------------------------------------------------------------
# T6 — chained decode (return_cache=True passthrough) yields same output
# ---------------------------------------------------------------------------
def test_decode_passthrough_cache():
    module = make_module()
    B, D, ctx, pred, P = 2, 1, 5, 3, module.patch_size
    target, obs, sid, tid, vid, pmask, ctx_total = make_packed_inputs(
        B, D, ctx, pred, P, device
    )

    with torch.no_grad():
        _, cache0 = module(
            target, obs, sid, tid, vid, pmask,
            training_mode=False, return_cache=True,
        )
        out_a, cache1 = module(
            target[:, ctx_total:],
            obs[:, ctx_total:],
            sid[:, ctx_total:],
            tid[:, ctx_total:],
            vid[:, ctx_total:],
            pmask[:, ctx_total:],
            training_mode=False, past_cache=cache0, return_cache=True,
        )
        out_b = module(
            target[:, ctx_total:],
            obs[:, ctx_total:],
            sid[:, ctx_total:],
            tid[:, ctx_total:],
            vid[:, ctx_total:],
            pmask[:, ctx_total:],
            training_mode=False, past_cache=cache1,
        )

    assert torch.allclose(out_a, out_b, atol=ATOL), \
        f"passthrough cache changed outputs: {(out_a - out_b).abs().max().item():.2e}"
    # Identity check — chained cache is the same dict by reference.
    assert cache1 is cache0, "decode should pass through the same cache object"
    print("✓ test_decode_passthrough_cache")

test_decode_passthrough_cache()

✓ test_decode_passthrough_cache
